In [8]:
import numpy as np
import scanpy as sc
import torch
from torch.utils.data import Dataset, Sampler, DataLoader

path = "/project/Wellcome_Discovery/datashare/lturiano/data/"

In [9]:
class DualAnnDataDataset(Dataset):
    def __init__(self, adata1, adata2, cell_type_key):
        """
        Parameters:
         - adata1, adata2: AnnData objects with the same number of cells.
         - cell_type_key: key in adata.obs that identifies cell populations.
        """
        if adata1.shape[0] != adata2.shape[0]:
            raise ValueError("Both AnnData objects must have the same number of cells.")
        
        self.adata1 = adata1
        self.adata2 = adata2
        # Assumes both adata have the same cell order for the given cell_type_key.
        self.cell_types = adata1.obs[cell_type_key].values
        
    def __len__(self):
        return self.adata1.shape[0]
    
    def __getitem__(self, idx):
        # Get expression data for cell at index idx.
        x1 = self.adata1.X[idx]
        x2 = self.adata2.X[idx]
        
        # If stored as sparse, convert to dense.
        if hasattr(x1, "toarray"):
            x1 = x1.toarray().squeeze()
        if hasattr(x2, "toarray"):
            x2 = x2.toarray().squeeze()
        
        # Convert to torch tensors.
        x1 = torch.tensor(x1, dtype=torch.float)
        x2 = torch.tensor(x2, dtype=torch.float)
        cell_type = self.cell_types[idx]
        return x1, x2, cell_type

In [10]:
rna_aligned_filt = sc.read_h5ad(path + "rna_filt_aligned.h5ad")
gex_aligned_filt = sc.read_h5ad(path + "gex_filt_aligned.h5ad")

In [11]:
duo = DualAnnDataDataset(rna_aligned_filt, gex_aligned_filt, "cell_type")

In [16]:
x1, x2, ct = duo.__getitem__(65000)

In [17]:
x1, x2, ct

(tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 1.3291]),
 tensor([0.0000, 0.0000, 0.0000,  ..., 0.9387, 0.0000, 0.0000]),
 'myeloid cell')

In [18]:
class CellTypeBatchSampler(Sampler):
    def __init__(self, dataset, batch_size):
        """
        Groups indices by cell type so that each batch is composed
        entirely of cells from one cell population.
        
        Parameters:
         - dataset: Instance of DualAnnDataDataset.
         - batch_size: Desired batch size.
        """
        self.batch_size = batch_size
        
        # Build a dictionary mapping cell type to list of indices.
        self.cell_type_to_indices = {}
        for idx, ct in enumerate(dataset.cell_types):
            self.cell_type_to_indices.setdefault(ct, []).append(idx)
        
        # Optionally, shuffle indices within each cell type.
        for ct in self.cell_type_to_indices:
            np.random.shuffle(self.cell_type_to_indices[ct])
        
        # Create a list of (cell type, indices) pairs and shuffle the order of cell types.
        self.groups = list(self.cell_type_to_indices.items())
        np.random.shuffle(self.groups)
        
        # Pre-calculate the batches per group.
        self.batches = []
        for ct, indices in self.groups:
            # Split indices for this cell type into chunks.
            for i in range(0, len(indices), self.batch_size):
                batch = indices[i : i + self.batch_size]
                self.batches.append(batch)
                
        # Shuffle the order of batches so that cell type groups are not strictly sequential.
        np.random.shuffle(self.batches)
    
    def __iter__(self):
        # Yield one batch (list of indices) at a time.
        for batch in self.batches:
            yield batch
    
    def __len__(self):
        return len(self.batches)

In [19]:
samp = CellTypeBatchSampler(duo, 64)
b1 = samp.__iter__()

In [20]:
b1

<generator object CellTypeBatchSampler.__iter__ at 0x79ed105e2260>